In [4]:
!pip install faiss-cpu groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.3 MB/s eta 0:00:00


In [5]:
import pandas as pd
import faiss # vector database
import numpy as np
from groq import Groq
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import os

In [6]:
from getpass import getpass
from groq import Groq

groq_api_key = getpass("Enter your Groq API Key: ")
client = Groq(api_key=groq_api_key)

Enter your Groq API Key: ··········


In [7]:
url = 'https://raw.githubusercontent.com/DataScience75/Top_mentor_projects_Datasets/refs/heads/main/customer_reviews.csv'
df = pd.read_csv(url)
df.head(5)

,review_id,product_id,review_text,rating
0,1,P101,Excellent battery life and superb camera quali...,5
1,2,P101,"The phone lasts two days on a single charge, g...",5
2,3,P101,Battery is good but the screen is average.,4
3,4,P101,Amazing camera but the design feels outdated.,4
4,5,P101,Very reliable battery performance.,5


In [12]:
# Task - Cleaning Data
### Chunk data in batches
### Create Embedding
def chunk_text(text,chunk_size = 10):
    words = text.split()
    #print(words)
    return [" ".join(words[i:i+chunk_size])for i in range(0,len(words),chunk_size)]

In [13]:
chunks = []
metadata = []
for idx, row in df.iterrows():
  for chunk in chunk_text(row['review_text']):
    chunks.append(chunk)
    metadata.append({'product_id':row['product_id'], 'review_id':row['review_id']})

In [15]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks, convert_to_numpy = True)
embeddings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

array([[-0.12552297,  0.04390913, -0.04582998, ...,  0.01603819,
        -0.07287003,  0.06687099],
       [ 0.04330344, -0.00582579, -0.00239526, ...,  0.03256203,
         0.01379364,  0.01132801],
       [-0.03790097,  0.04702243,  0.05273018, ...,  0.0315903 ,
        -0.02566351,  0.05701391],
       ...,
       [ 0.03489391, -0.04569964,  0.01673001, ...,  0.02031808,
        -0.12329303,  0.07027457],
       [ 0.04399861, -0.01214795,  0.09306977, ..., -0.03072306,
        -0.0578397 ,  0.02898447],
       [-0.01532622,  0.03267722,  0.04441229, ..., -0.06343682,
        -0.09644358,  0.08354177]], dtype=float32)

In [16]:
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
# FAISS index initialized

In [21]:

#RAG - KNOWLEDGE BASE (embeddings + product_id, review_id)
## Create the LLM
### Prepare the context properly using metadata and relevant information
####  Pass that to the LLM for precise results
##### Recommendation Engine - Goal

def recommend_product(user_query, top_k=3):
    query_vec = model.encode([user_query], convert_to_numpy=True)
    distance, indices = index.search(query_vec, top_k)

    # Retrieval Relevant reviews based on index
    retrieved_chunks = [chunks[i] for i in indices[0]]
    retrieved_metadata = [metadata[i] for i in indices[0]]

    context = "\n".join(
        [f"product {m['product_id']}: {txt}" for txt, m in zip(retrieved_chunks, retrieved_metadata)]
    )

    print(context)

    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful product recommendation assistant"
            },
            {
                "role": "user",
                "content": f"Based on the following customer reviews:\n{context}\n\nSuggest the best product for {user_query}"
            }
        ],
        temperature=0.2,
        max_tokens=200
    )

    return completion.choices[0].message.content, retrieved_metadata

In [22]:
recommendations = recommend_product("I need a light weight camera phone")
print("Find the recommendations - ", recommendations)

product P105: Budget-friendly but camera is basic.
product P101: Amazing camera but the design feels outdated.
product P105: Low cost and works well for calls.
Find the recommendations -  ('Based on the customer reviews, I would suggest product P105 as the best option for a lightweight camera phone. Although the camera is described as "basic", the customer reviews also mention that the product is "budget-friendly" and "works well for calls", which suggests that it may be a good option for someone who wants a lightweight camera phone without breaking the bank.\n\nAdditionally, the fact that the product is described as "lightweight" is not mentioned in the reviews for product P101, so it\'s possible that product P105 may be the more suitable option for someone who wants a lightweight camera phone.\n\nHowever, if you\'re looking for a camera phone with a high-quality camera, product P101 may still be a good option, even if the design is outdated. It ultimately depends on your priorities a